In [8]:
# -------------------------------------------------------------
# 00_preprocess.ipynb – Environment setup & bilingual chunking
# -------------------------------------------------------------
import os
import sys
import subprocess
import json
from pathlib import Path
import yaml

# -------------------------------------------------------------
# 1️ Load config.yaml
# -------------------------------------------------------------
config = yaml.safe_load(open("config.yaml", "r", encoding="utf-8"))

CHUNK_SIZE = config["chunking"]["size"]
CHUNK_OVERLAP = config["chunking"]["overlap"]

DATA_DIR = Path(config["paths"]["data"]).resolve()
MASTER_FILE = DATA_DIR / "master.jsonl"
CHUNKS_FILE = DATA_DIR / "chunks.jsonl"

print("Chunk size:", CHUNK_SIZE)
print("Overlap:", CHUNK_OVERLAP)
print("Master JSONL:", MASTER_FILE)
print("Chunks output:", CHUNKS_FILE)

# -------------------------------------------------------------
# 2️ Language extraction utilities
# -------------------------------------------------------------
def extract_lang(val, lang):
    """Return only specified language text from dict/list/string, always as string."""
    if isinstance(val, dict):
        return str(val.get(lang, ""))
    if isinstance(val, list):
        return ", ".join([extract_lang(x, lang) for x in val])
    return str(val)

def text_from_fields(obj, lang="en"):
    """Flatten disease record into a readable text block."""
    sections = []
    field_order = [
        "description",
        "symptoms",
        "causes",
        "samprapti",
        "treatments",
        "dietary_recommendations",
        "lifestyle_recommendations",
        "precautions",
        "prognosis"
    ]
    for key in field_order:
        if key not in obj:
            continue
        val = obj[key]
        if isinstance(val, dict) or isinstance(val, list):
            sections.append(extract_lang(val, lang))
        else:
            sections.append(str(val))
    # force string conversion to avoid AttributeError
    text = "\n".join([str(s) for s in sections if str(s).strip()])
    return text

# -------------------------------------------------------------
# 3️ Chunking function (word-based, with overlap)
# -------------------------------------------------------------
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = end - overlap
        if start < 0:
            start = 0
    return chunks

# -------------------------------------------------------------
# 4️ Load master.jsonl
# -------------------------------------------------------------
records = []
with open(MASTER_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print("Loaded records:", len(records))

# -------------------------------------------------------------
# 5️ Process → flatten → chunk → build chunk rows
# -------------------------------------------------------------
chunk_rows = []
chunk_id = 0

for item in records:
    disease_id = item["disease_id"]
    disease_name_en = item.get("disease_name", {}).get("en")
    disease_name_si = item.get("disease_name", {}).get("si")

    for lang in ("en", "si"):
        text = text_from_fields(item, lang=lang)
        if not text.strip():
            continue
        chunks = chunk_text(text)
        for ch in chunks:
            chunk_rows.append({
                "chunk_id": f"{disease_id}_{chunk_id}",
                "doc_id": disease_id,
                "disease_name_en": disease_name_en,
                "disease_name_si": disease_name_si,
                "lang": lang,
                "text": ch,
                "source": item.get("metadata", {}).get("origin", ""),
                "fields_present": list(item.keys())
            })
            chunk_id += 1

print("Total chunks:", len(chunk_rows))

# -------------------------------------------------------------
# 6️ Save chunks.jsonl
# -------------------------------------------------------------
with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    for row in chunk_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved chunks.jsonl →", CHUNKS_FILE)
print("Preprocessing complete!")

Chunk size: 512
Overlap: 50
Master JSONL: E:\Uni Videos\Academic Notes\4th YEAR\2nd Sem\Research\data\master.jsonl
Chunks output: E:\Uni Videos\Academic Notes\4th YEAR\2nd Sem\Research\data\chunks.jsonl
Loaded records: 10
Total chunks: 20
Saved chunks.jsonl → E:\Uni Videos\Academic Notes\4th YEAR\2nd Sem\Research\data\chunks.jsonl
Preprocessing complete!
